# Types of AI: a hands-on tour

Companion notebook for the lesson **Types of AI** (Track 01, Module 1.1).

**What you'll learn**

1. The two common ways to classify AI: by **capability** (Narrow, General, Super) and by **functionality** (Reactive, Limited memory, Theory of mind, Self-aware).
2. Which of those types actually **exist today** (spoiler: only two of the seven).
3. The real difference between a **reactive** machine and a **limited-memory** machine, using two tiny thermostats.
4. How to place a real system, such as a spam filter or a chatbot, on both scales.

Everything runs locally in plain Python (standard library only). No API keys, no downloads. All the code is in this notebook: run the cells from top to bottom.

## Step 1: Describe one type of AI

We need a small, fixed record for each type of AI: its name, which scale (axis) it belongs to, its level on that scale, a short description, whether it exists today, and an example. A frozen `dataclass` is a neat way to write such a record in Python.

We also write a tiny helper, `format_table`, that prints rows as a lined-up text table. It is plain string formatting, nothing AI-specific.

In [1]:
from collections import deque
from dataclasses import dataclass


@dataclass(frozen=True)
class AIType:
    """One type of AI in one of the two classifications."""

    name: str
    axis: str  # "capability" or "functionality"
    level: int  # 1 = simplest
    description: str
    exists_today: bool
    example: str


def catalogue_rows(types):
    """Return (name, exists today?, example) rows ready to print."""
    return [(t.name, "yes" if t.exists_today else "no", t.example) for t in types]


def format_table(rows, headers):
    """Format rows as a simple fixed-width text table."""
    all_rows = [headers, *rows]
    widths = [max(len(str(row[i])) for row in all_rows) for i in range(len(headers))]
    lines = []
    for index, row in enumerate(all_rows):
        lines.append("  ".join(str(cell).ljust(width) for cell, width in zip(row, widths)).rstrip())
        if index == 0:
            lines.append("  ".join("-" * width for width in widths))
    return "\n".join(lines)


print(format_table([("a", "bb")], ("H1", "H2")))

H1  H2
--  --
a   bb


The little test table shows the layout: a header row, a dashed line, then one line per row, with columns padded to the same width. We'll use it for every table below.

## Step 2: Classification by capability

The first scale asks: **how broad is the system's intelligence?**

- **Narrow AI** is good at one task (or a small family of tasks).
- **General AI** (often called AGI) would learn and reason across any task a human can.
- **Super AI** would beat the best humans at almost everything.

Let's write the catalogue and print it, including whether each type exists today.

In [2]:
CAPABILITY_TYPES = [
    AIType("Narrow AI", "capability", 1,
           "Does one task, or a small family of tasks, well.",
           True, "Spam filter, face unlock, chess engine, chatbot"),
    AIType("General AI", "capability", 2,
           "Learns and reasons across any task a human can, without retraining.",
           False, "None yet (research goal)"),
    AIType("Super AI", "capability", 3,
           "Exceeds the best humans at almost everything.",
           False, "None (hypothetical)"),
]

print(format_table(catalogue_rows(CAPABILITY_TYPES), ("Capability type", "Exists today?", "Example")))

Capability type  Exists today?  Example
---------------  -------------  -----------------------------------------------
Narrow AI        yes            Spam filter, face unlock, chess engine, chatbot
General AI       no             None yet (research goal)
Super AI         no             None (hypothetical)


**What this tells us:** every AI system in use today, including the most impressive chatbots, is classed as **Narrow AI** by sources such as IBM. General and Super AI are research goals or thought experiments. The next lesson, *Narrow AI vs General AI*, looks at this line in depth.

## Step 3: Classification by functionality

The second scale asks a different question: **how does the system work with information over time?** Does it remember? Does it understand other minds?

In [3]:
FUNCTIONALITY_TYPES = [
    AIType("Reactive machine", "functionality", 1,
           "Reacts to the current input only. No memory of the past.",
           True, "IBM Deep Blue, a simple thermostat rule"),
    AIType("Limited memory", "functionality", 2,
           "Uses recent or stored past data to decide.",
           True, "Self-driving perception, recommenders, chatbots"),
    AIType("Theory of mind", "functionality", 3,
           "Understands that others have beliefs, goals and emotions.",
           False, "None yet (research goal)"),
    AIType("Self-aware", "functionality", 4,
           "Has a sense of its own existence and inner state.",
           False, "None (hypothetical)"),
]

print(format_table(catalogue_rows(FUNCTIONALITY_TYPES), ("Functionality type", "Exists today?", "Example")))

Functionality type  Exists today?  Example
------------------  -------------  -----------------------------------------------
Reactive machine    yes            IBM Deep Blue, a simple thermostat rule
Limited memory      yes            Self-driving perception, recommenders, chatbots
Theory of mind      no             None yet (research goal)
Self-aware          no             None (hypothetical)


**What this tells us:** only the first two levels exist. *Theory of mind* and *self-aware* AI are still ideas. A chatbot that *sounds* empathetic is predicting likely text; it is not shown to have a model of your beliefs or a sense of itself.

The two scales are independent: one system gets **one label on each scale**.

### Quick check: what exists today?

Let's add a small lookup, `exists_today`, and confirm with an `assert` that exactly three of the seven types exist today. An `assert` stays silent when it is true and stops the notebook with an error when it is false.

In [4]:
def exists_today(type_name):
    """True if systems of this type exist today."""
    for ai_type in CAPABILITY_TYPES + FUNCTIONALITY_TYPES:
        if ai_type.name == type_name:
            return ai_type.exists_today
    raise KeyError(f"Unknown AI type: {type_name!r}")


existing = {t.name for t in CAPABILITY_TYPES + FUNCTIONALITY_TYPES if t.exists_today}
assert existing == {"Narrow AI", "Reactive machine", "Limited memory"}
print("Types that exist today:", sorted(existing))

Types that exist today: ['Limited memory', 'Narrow AI', 'Reactive machine']


Narrow AI on the first scale, and reactive and limited-memory machines on the second. Everything else is a research goal or hypothetical. Next, let's see the two functionality levels that do exist in action.

## Step 4: A reactive machine

A **reactive machine** looks only at the current input. It has no memory, so **the same input always gives the same output**.

Our `ReactiveThermostat` aims for 21 °C with a tolerance of 1 °C: above 22 it cools, below 20 it heats, otherwise it stays idle.

In [5]:
class ReactiveThermostat:
    """A reactive machine: the decision depends on the current reading only."""

    def __init__(self, target=21.0, tolerance=1.0):
        self.target = target
        self.tolerance = tolerance

    def decide(self, temperature):
        if temperature > self.target + self.tolerance:
            return "cool"
        if temperature < self.target - self.tolerance:
            return "heat"
        return "idle"


reactive = ReactiveThermostat(target=21.0, tolerance=1.0)

for reading in [18, 21, 22, 23, 22]:
    print(f"{reading} °C -> {reactive.decide(reading)}")

18 °C -> heat
21 °C -> idle
22 °C -> idle
23 °C -> cool
22 °C -> idle


Notice that 22 °C gives `idle` both times. The thermostat cannot tell whether the room is warming up or cooling down, because it never looks back. That is the defining limit of a reactive machine. IBM's Deep Blue chess computer worked the same way in spirit: it evaluated the board in front of it without learning from past games during play.

## Step 5: A limited-memory machine

A **limited-memory** machine keeps some recent data and uses it to decide. Our `LimitedMemoryThermostat` builds on the reactive one. It remembers the last few readings in a `deque` with a maximum length (a list that drops its oldest item when full), estimates the trend, predicts the next temperature, and acts on the *prediction*.

In [6]:
class LimitedMemoryThermostat(ReactiveThermostat):
    """A limited-memory machine: remembers the last few readings.

    It uses them to predict the next temperature and acts on that prediction.
    Older readings fall out of memory (that is the "limited" part).
    """

    def __init__(self, target=21.0, tolerance=1.0, memory_size=3):
        super().__init__(target, tolerance)
        if memory_size < 2:
            raise ValueError("memory_size must be at least 2 to see a trend")
        self.memory = deque(maxlen=memory_size)

    def predict_next(self):
        """Last reading plus the average change between remembered readings."""
        readings = list(self.memory)
        if len(readings) < 2:
            return readings[-1]
        average_change = (readings[-1] - readings[0]) / (len(readings) - 1)
        return readings[-1] + average_change

    def decide(self, temperature):
        self.memory.append(temperature)
        return super().decide(self.predict_next())

Now we feed both thermostats the same two sequences that both end at 22 °C: one where the room is warming, one where it is cooling.

In [7]:
warming = [18, 19, 20, 21, 22]
cooling = [26, 25, 24, 23, 22]

for label, readings in [("Warming room", warming), ("Cooling room", cooling)]:
    reactive = ReactiveThermostat()
    memory = LimitedMemoryThermostat(memory_size=3)
    print(label)
    for r in readings:
        print(f"  {r} °C   reactive: {reactive.decide(r):<5}  limited memory: {memory.decide(r):<5}  "
              f"(predicted next: {memory.predict_next():.1f} °C)")
    print()

Warming room
  18 °C   reactive: heat   limited memory: heat   (predicted next: 18.0 °C)
  19 °C   reactive: heat   limited memory: idle   (predicted next: 20.0 °C)
  20 °C   reactive: idle   limited memory: idle   (predicted next: 21.0 °C)
  21 °C   reactive: idle   limited memory: idle   (predicted next: 22.0 °C)
  22 °C   reactive: idle   limited memory: cool   (predicted next: 23.0 °C)

Cooling room
  26 °C   reactive: cool   limited memory: cool   (predicted next: 26.0 °C)
  25 °C   reactive: cool   limited memory: cool   (predicted next: 24.0 °C)
  24 °C   reactive: cool   limited memory: cool   (predicted next: 23.0 °C)
  23 °C   reactive: cool   limited memory: idle   (predicted next: 22.0 °C)
  22 °C   reactive: idle   limited memory: idle   (predicted next: 21.0 °C)



**Interpreting the result:**

- At the final reading of 22 °C, the reactive thermostat says `idle` in both rooms.
- The limited-memory thermostat says `cool` in the warming room (it predicts 23 °C next) and `idle` in the cooling room (it predicts 21 °C).
- It also stops heating and stops cooling *earlier*, because it sees where the temperature is heading.

Same current input, different decision, because of memory. This is the same idea, at a tiny scale, as a self-driving car estimating another car's speed from its last few positions.

## Step 6: Why "limited"?

The memory has a fixed size. Older readings are forgotten. Let's look inside.

In [8]:
memory = LimitedMemoryThermostat(memory_size=3)
for r in [10, 15, 20, 21, 22]:
    memory.decide(r)
    print(f"after {r:>2} °C, memory holds: {list(memory.memory)}")

assert list(memory.memory) == [20, 21, 22]  # quick check: only the last 3 readings are kept

after 10 °C, memory holds: [10]
after 15 °C, memory holds: [10, 15]
after 20 °C, memory holds: [10, 15, 20]
after 21 °C, memory holds: [15, 20, 21]
after 22 °C, memory holds: [20, 21, 22]


The readings 10 and 15 have dropped out. Real limited-memory systems work the same way: a chatbot has a fixed context window, a car keeps a short history of sensor frames, and a trained model's knowledge is frozen at its training data until it is retrained.

## Step 7: Classify real systems on both scales

Now we describe a system with simple yes/no facts in a `SystemProfile`, and write one small function per scale. Each function checks the most advanced level first and falls back to the simplest one, so a system gets the **highest** level it reaches.

In [9]:
@dataclass(frozen=True)
class SystemProfile:
    """Yes/no facts about an AI system, used to place it on both axes."""

    name: str
    uses_past_data: bool = False
    models_other_minds: bool = False
    is_self_aware: bool = False
    general_across_domains: bool = False
    beats_humans_at_everything: bool = False


def classify_capability(profile):
    if profile.general_across_domains and profile.beats_humans_at_everything:
        return "Super AI"
    if profile.general_across_domains:
        return "General AI"
    return "Narrow AI"


def classify_functionality(profile):
    """Return the highest functionality level the profile reaches."""
    if profile.is_self_aware:
        return "Self-aware"
    if profile.models_other_minds:
        return "Theory of mind"
    if profile.uses_past_data:
        return "Limited memory"
    return "Reactive machine"


def classify(profile):
    """Return (name, capability type, functionality type) for one system."""
    return profile.name, classify_capability(profile), classify_functionality(profile)

Let's describe four real systems and place each one on both scales.

In [10]:
EXAMPLE_SYSTEMS = [
    SystemProfile("IBM Deep Blue (1997 chess)"),
    SystemProfile("Email spam filter", uses_past_data=True),
    SystemProfile("Self-driving car perception", uses_past_data=True),
    SystemProfile("LLM chatbot (e.g. ChatGPT)", uses_past_data=True),
]

rows = [classify(system) for system in EXAMPLE_SYSTEMS]
print(format_table(rows, ("System", "Capability", "Functionality")))

System                       Capability  Functionality
---------------------------  ----------  ----------------
IBM Deep Blue (1997 chess)   Narrow AI   Reactive machine
Email spam filter            Narrow AI   Limited memory
Self-driving car perception  Narrow AI   Limited memory
LLM chatbot (e.g. ChatGPT)   Narrow AI   Limited memory


Every real system lands in **Narrow AI + Reactive** or **Narrow AI + Limited memory**. That is the honest picture of AI today. Whether large language models are still "narrow" is debated by researchers because they handle many tasks; most mainstream sources still class them as narrow, and the next lesson covers that debate.

## Step 8: Try a hypothetical system

What if someone claims to have built a robot that understands how you feel? Describe it and see where it lands, then ask whether that type exists today.

In [11]:
claimed_robot = SystemProfile("Empathic care robot (claimed)", uses_past_data=True, models_other_minds=True)
name, capability, functionality = classify(claimed_robot)
print(f"{name}: {capability} + {functionality}")
print(f"Does '{functionality}' AI exist today? {exists_today(functionality)}")

Empathic care robot (claimed): Narrow AI + Theory of mind
Does 'Theory of mind' AI exist today? False


The classifier puts the claim at *Theory of mind*, a type that does not exist yet. So a sensible engineer treats the claim with care: most likely the robot **imitates** empathy (for example, by detecting sentiment in your words) rather than truly modelling your mind. Being able to spot that gap is the practical value of knowing these categories.

## Recap

- **Two scales, one label on each.** Capability: Narrow → General → Super. Functionality: Reactive → Limited memory → Theory of mind → Self-aware.
- **What exists today:** Narrow AI, and within it reactive and limited-memory systems. General, Super, Theory of mind and Self-aware AI do not exist yet.
- **Reactive vs limited memory:** a reactive machine gives the same output for the same input; a limited-memory machine uses recent history, so its decision can change.
- **Memory is limited:** fixed windows, frozen training data and context limits all mean old information is dropped.
- **AI agents** (what this academy builds) are Narrow AI with limited memory: an LLM plus tools, plus memory you design.

Try changing `memory_size`, the target temperature, or the readings, and re-run the cells to see how the decisions change.